In [ ]:
!pip -q install librosa soundfile

import librosa
import numpy as np
import matplotlib.pyplot as plt

from google.colab import files
from IPython.display import display, HTML



display(HTML("""
<div style="
background:#f5f7fb;
padding:30px;
border-radius:20px;
text-align:center;
font-family:Arial;
">

<h1>Speech Emotion Analyzer</h1>

<p style="font-size:18px;">
Analyze speech characteristics and estimate the speaker's emotion
</p>

<p>
Text & Speech Analysis - Application 6
</p>

</div>
"""))


# ============================================================
# UPLOAD AUDIO
# ============================================================

print("\nUpload a WAV audio recording.")

print("""
For better results:
- Speak for around 5–10 seconds
- Use a clear recording
- Record in a quiet environment
- Speak naturally
""")

uploaded = files.upload()

audio_file = list(uploaded.keys())[0]

print("\nUploaded file:")
print(audio_file)


# ============================================================
# LOAD AUDIO
# ============================================================

print("\nAnalyzing audio...")

audio, sample_rate = librosa.load(
    audio_file,
    sr=None,
    mono=True
)


# ============================================================
# AUDIO DURATION
# ============================================================

duration = librosa.get_duration(
    y=audio,
    sr=sample_rate
)


# ============================================================
# RMS ENERGY
# ============================================================

rms = librosa.feature.rms(
    y=audio
)

average_energy = float(
    np.mean(rms)
)


# ============================================================
# ZERO CROSSING RATE
# ============================================================

zcr = librosa.feature.zero_crossing_rate(
    audio
)

average_zcr = float(
    np.mean(zcr)
)


# ============================================================
# PITCH ESTIMATION
# ============================================================

try:

    f0, voiced_flag, voiced_probabilities = (
        librosa.pyin(
            audio,
            fmin=librosa.note_to_hz('C2'),
            fmax=librosa.note_to_hz('C7')
        )
    )

    pitch_values = f0[~np.isnan(f0)]

    if len(pitch_values) > 0:

        average_pitch = float(
            np.mean(pitch_values)
        )

    else:

        average_pitch = 0

except:

    average_pitch = 0


# ============================================================
# SPEECH RATE APPROXIMATION
# ============================================================

onset_env = librosa.onset.onset_strength(
    y=audio,
    sr=sample_rate
)

onset_frames = librosa.onset.onset_detect(
    onset_envelope=onset_env,
    sr=sample_rate
)

number_of_onsets = len(onset_frames)

if duration > 0:

    speech_activity_rate = (
        number_of_onsets / duration
    )

else:

    speech_activity_rate = 0


# ============================================================
# NORMALIZED FEATURES
# ============================================================

energy_score = min(
    average_energy * 1000,
    100
)

pitch_score = min(
    average_pitch / 5,
    100
)

zcr_score = min(
    average_zcr * 1000,
    100
)


# ============================================================
# EMOTION ESTIMATION
# ============================================================

# Rule-based educational model

if (
    average_energy > 0.08
    and average_pitch > 180
):

    emotion = "Happy / Excited"

    explanation = (
        "Higher energy and pitch suggest an energetic "
        "or excited speaking style."
    )

elif (
    average_energy > 0.07
    and average_pitch < 160
):

    emotion = "Angry / Strong"

    explanation = (
        "Higher vocal energy combined with a lower pitch "
        "may indicate a strong or angry speaking style."
    )

elif (
    average_energy < 0.03
    and average_pitch < 160
):

    emotion = "Sad / Calm"

    explanation = (
        "Lower energy and pitch suggest a calm or "
        "possibly sad speaking style."
    )

elif (
    average_energy < 0.05
):

    emotion = "Calm / Neutral"

    explanation = (
        "The speech contains relatively low and stable "
        "energy characteristics."
    )

else:

    emotion = "Neutral"

    explanation = (
        "The detected audio characteristics do not strongly "
        "indicate a specific emotion."
    )


# ============================================================
# CONFIDENCE ESTIMATION
# ============================================================

if average_energy > 0 and average_pitch > 0:

    confidence = min(
        60 + abs(average_energy * 300),
        95
    )

else:

    confidence = 50


# ============================================================
# DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 65)

print("SPEECH EMOTION ANALYSIS")

print("=" * 65)

print(f"\nAudio Duration       : {duration:.2f} seconds")

print(
    f"Average Energy      : "
    f"{average_energy:.4f}"
)

print(
    f"Average Pitch       : "
    f"{average_pitch:.2f} Hz"
)

print(
    f"Zero Crossing Rate  : "
    f"{average_zcr:.4f}"
)

print(
    f"Speech Activity     : "
    f"{speech_activity_rate:.2f}"
)

print("\nEstimated Emotion:")
print(emotion)

print(
    f"\nEstimated Confidence: "
    f"{confidence:.2f}%"
)

print("\nExplanation:")
print(explanation)


# ============================================================
# VISUAL DASHBOARD
# ============================================================

display(HTML(f"""

<div style="
max-width:850px;
margin:30px auto;
font-family:Arial;
">

<div style="
background:white;
padding:30px;
border-radius:20px;
border:1px solid #ddd;
">

<h2 style="text-align:center;">
Speech Emotion Analysis Result
</h2>

<hr>

<div style="
text-align:center;
padding:30px;
background:#f5f7fb;
border-radius:15px;
">

<h1>
{emotion}
</h1>

<p style="font-size:20px;">
Confidence: <b>{confidence:.2f}%</b>
</p>

</div>

<br>

<h3>Audio Features</h3>

<table style="
width:100%;
border-collapse:collapse;
text-align:center;
">

<tr>

<th style="padding:12px;border:1px solid #ddd;">
Feature
</th>

<th style="padding:12px;border:1px solid #ddd;">
Value
</th>

</tr>

<tr>

<td style="padding:10px;border:1px solid #ddd;">
Duration
</td>

<td style="padding:10px;border:1px solid #ddd;">
{duration:.2f} seconds
</td>

</tr>

<tr>

<td style="padding:10px;border:1px solid #ddd;">
Average Energy
</td>

<td style="padding:10px;border:1px solid #ddd;">
{average_energy:.4f}
</td>

</tr>

<tr>

<td style="padding:10px;border:1px solid #ddd;">
Average Pitch
</td>

<td style="padding:10px;border:1px solid #ddd;">
{average_pitch:.2f} Hz
</td>

</tr>

<tr>

<td style="padding:10px;border:1px solid #ddd;">
Zero Crossing Rate
</td>

<td style="padding:10px;border:1px solid #ddd;">
{average_zcr:.4f}
</td>

</tr>

</table>

<br>

<h3>Analysis</h3>

<p>
{explanation}
</p>

</div>

</div>

"""))



plt.figure(figsize=(12, 4))

librosa.display.waveshow(
    audio,
    sr=sample_rate
)

plt.title("Speech Waveform")

plt.xlabel("Time (seconds)")

plt.ylabel("Amplitude")

plt.show()


# ============================================================
# SPECTROGRAM
# ============================================================

spectrogram = librosa.feature.melspectrogram(
    y=audio,
    sr=sample_rate
)

spectrogram_db = librosa.power_to_db(
    spectrogram,
    ref=np.max
)

plt.figure(figsize=(12, 5))

librosa.display.specshow(
    spectrogram_db,
    sr=sample_rate,
    x_axis="time",
    y_axis="mel"
)

plt.title("Speech Spectrogram")

plt.colorbar(format="%+2.0f dB")

plt.show()


Upload a WAV audio recording.

For better results:
- Speak for around 5–10 seconds
- Use a clear recording
- Record in a quiet environment
- Speak naturally

